## Show, Attend and Tell — Training (Flickr8k)


In [ ]:
# Reinstall ipython kernel to enable autoreload on latest runtime (by 02/12/2026)
# You'll be prompted to restart the session.
!pip install ipython==8.12.0

In [ ]:
%load_ext autoreload
%autoreload 2

Clone repository

In [ ]:
!git clone https://github.com/seanzhangw/show-attend-tell.git

Pull recent changes

In [ ]:
%cd /content/show-attend-tell
!git pull

Get dataset

In [ ]:
%cd /content/show-attend-tell/data
!python get_flickr8k.py

Imports

In [ ]:
%cd /content/show-attend-tell/code

import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

import config
from datasets.flickr8k import collate_fn, build_flickr8k_dataset_split
from eval.greedy_decode import greedy_decode
from models.encoder import EncoderCNN
from models.decoder import Decoder


Define Hyperparameters

In [ ]:
# Hyperparameters (edit as needed)
epochs = 8
batch_size = config.BATCH_SIZE
lr = 1e-4
embed_dim = 512
hidden_dim = 512
attention_dim = 512

dropout = 0.5
lambda_att = 1.0
grad_clip = 5.0
val_ratio = 0.1
split_seed = 42
# BLEU-4: cap images per eval for speed (None = all val images)
bleu_max_images = 500

Build datasets, dataloaders

In [ ]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            # ImageNet normalization for pre-trained ResNet (ImageNet dataset mean and std)
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

# Dataset: image-level train/val split; vocab from train captions only
(
    train_set,
    val_set,
    val_image_ids,
    full_captions_map,
    word2idx,
    idx2word,
) = build_flickr8k_dataset_split(
    config,
    transform=transform,
    val_ratio=val_ratio,
    seed=split_seed,
)
pad_idx = word2idx["<pad>"]
print(
    f"Train samples: {len(train_set)} | Val samples: {len(val_set)} | "
    f"Val images: {len(val_image_ids)} | Vocab size: {len(word2idx)}"
)

train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_set,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
)

Initialize models, loss, and optimizer

In [ ]:
# Models
encoder = EncoderCNN().to(device)
decoder = Decoder(
    vocab_size=len(word2idx),
    embed_dim=embed_dim,
    feature_dim=2048,
    hidden_dim=hidden_dim,
    attention_dim=attention_dim,
    dropout=dropout,
).to(device)

# Loss / Optimizer
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

# Only decoder trained since encoder uses resnet50
params = list(decoder.parameters())

optimizer = torch.optim.Adam(params, lr=lr)

Training loop. Current config starts training from scratch

In [ ]:
from training.loop import evaluate_caption_metrics, run_training_loop
from training.run_spec import CorpusEvalSpec, TrainLoopOptions

CHECKPOINT_DIR = "./checkpoints"
RESUME_PATH = "./checkpoints/best_by_val_loss.pt"  # e.g. "./checkpoints/best_by_val_loss.pt"
EVAL_METRICS = ["bleu1", "bleu2", "bleu3", "bleu4"]  # add "meteor" after nltk.download(...)
BEST_BY = "val_loss"  # best checkpoint by this key; min for train_loss/val_loss, max for BLEU etc.

corpus_eval = CorpusEvalSpec(
    val_image_ids=val_image_ids,
    full_captions_map=full_captions_map,
    image_dir=config.IMAGE_DIR,
    transform=transform,
    word2idx=word2idx,
    idx2word=idx2word,
    eval_metric_names=EVAL_METRICS,
    max_decode_len=config.MAX_LEN,
    max_images=bleu_max_images,
)

train_options = TrainLoopOptions(
    checkpoint_dir=CHECKPOINT_DIR,
    best_by=BEST_BY,
    resume_path=RESUME_PATH,
    lambda_att=lambda_att,
    grad_clip=grad_clip,
    print_every=100,
)

_, best_ckpt_path, _ = run_training_loop(
    encoder,
    decoder,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    epochs,
    corpus_eval,
    train_options,
    hparams={
        "embed_dim": embed_dim,
        "hidden_dim": hidden_dim,
        "attention_dim": attention_dim,
        "dropout": dropout,
        "lambda_att": lambda_att,
        "lr": lr,
        "batch_size": batch_size,
        "val_ratio": val_ratio,
        "split_seed": split_seed,
    },
)


Optional model evaluation (also runs in training loop)

In [ ]:
# Eval-only metrics (no training step)
metric_scores = evaluate_caption_metrics(encoder, decoder, device, corpus_eval)
print("Eval-only metrics:", metric_scores)

Eye Test - generate captions on validation images 

In [ ]:
from PIL import Image

num_examples = min(20, len(val_image_ids))
print(f"\nQualitative eval on {num_examples} validation images (unique)")

for i, img_name in enumerate(sorted(val_image_ids)[:num_examples]):
    path = os.path.join(config.IMAGE_DIR, img_name)
    image = Image.open(path).convert("RGB")
    image_tensor = transform(image)

    pred_tokens = greedy_decode(
        encoder,
        decoder,
        image_tensor,
        word2idx,
        idx2word,
        device,
        max_len=config.MAX_LEN,
    )
    refs = full_captions_map[img_name][:5]

    print(f"\n[{i+1:02d}] Image: {img_name}")
    print("Generated:", " ".join(pred_tokens) if pred_tokens else "<empty>")
    for j, ref in enumerate(refs[:3], start=1):
        print(f"Ref {j}: {ref}")


Save the trained model to Google Drive

Connects to google drive

In [ ]:
from google.colab import drive
import sys

drive.mount('/content/gdrive')

# NOTE: Change this path to your own Google Drive path
base_dir = "/content/gdrive/MyDrive/show-attend-tell"
sys.path.append(base_dir)

Saves the model file to drive

In [ ]:
import shutil
import os

# Define the source (where the model is now) and destination (your Drive)
source_path = './checkpoints/best_by_val_loss.pt'
dest_path = os.path.join(base_dir, 'best_by_val_loss_4_19.pt')

# Create the directory on Drive if it doesn't exist
os.makedirs(base_dir, exist_ok=True)

# Copy the file
shutil.copyfile(source_path, dest_path)

print(f"Model successfully saved to: {dest_path}")

Loading a model into encoder_loaded, decoder_loaded variables. If running the training loop, need to change variable names

In [ ]:
# Load encoder/decoder from a checkpoint stored on Google Drive
from training.checkpoint import load_models_from_checkpoint_path

# Path on Drive where we saved the checkpoint above
ckpt_path = os.path.join(base_dir, "best_by_val_loss.pt")
print("Loading checkpoint from:", ckpt_path)

# Re-create model objects with the same architecture as during training
encoder_loaded = EncoderCNN().to(device)
decoder_loaded = Decoder(
    vocab_size=len(word2idx),
    embed_dim=embed_dim,
    feature_dim=2048,
    hidden_dim=hidden_dim,
    attention_dim=attention_dim,
    dropout=dropout,
).to(device)

# (Optional) create an optimizer if you want to resume training
optimizer_loaded = torch.optim.Adam(list(decoder_loaded.parameters()), lr=lr)

resume_info = load_models_from_checkpoint_path(
    ckpt_path,
    encoder_loaded,
    decoder_loaded,
    optimizer=optimizer_loaded,  # set to None if you only care about inference
    map_location=device,
)

print("Start epoch after restore:", resume_info["start_epoch"])
print("Last metrics in checkpoint:", resume_info["metrics"])
print("Tracking info:", resume_info["tracking"])